# 🤖 Build Your Own RAG Chatbot from Scratch


## A Beginner-Friendly Guide (Zero to Working Chatbot)

    Welcome! Today, you are going to build a chatbot that doesn't just "guess" answers, 
    but actually reads documents to give you facts.

## 📚 What is a "Normal" Chatbot?

    A normal AI (like ChatGPT) is like a student who memorized a million books years ago 
    but isn't allowed to look at new notes. 
    If you ask it about something that happened yesterday or a private document you wrote, 
    it will hallucinate (confidently lie).

## 🔍 What is a RAG Chatbot?

    RAG stands for Retrieval-Augmented Generation.
    Think of it as giving the AI an "Open Book Exam." 
        1. Retrieval: The AI looks through your specific documents to find the right page.
        2. Augmented: It adds that information to your question.
        3. Generation: It writes a perfect answer based only on those facts.

### Flowchart of RAG

```mermaid
flowchart TD
    Q[User Question] --> EmbedQ[Embed];

    DB[Vector DB] -->|top-k| Retrieve[Retrieve relevant chunks];

    EmbedQ --> Retrieve;
    Retrieve --> Context[Augmented Prompt = query + chunks];

    Context --> LLM(LLM);
    LLM -->|generates| R[Answer];

In [ ]:
# STEP 1: INSTALLING OUR TOOLS
# We need to download some 'libraries' (pre-written code) to help us.

# 'langchain' is the main framework for building AI apps
!pip install langchain langchain-community langchain-huggingface 

# 'chromadb' is our "Vector Database" (the AI's library where it stores facts)
!pip install chromadb 

# 'sentence-transformers' helps the AI understand the "meaning" of sentences
!pip install sentence-transformers 

# 'pypdf' lets the AI read PDF files
!pip install pypdf 

# 'gradio' creates a pretty chat window for us to type in
!pip install gradio

!pip install ollama
print("✅ All tools installed successfully!")

## 🧪 Experiment 1: The "Normal" AI (No Context)

In [13]:
import ollama
question = f"""
            For the F/A-18 Hornet, below what aircraft gross weight does the Flight Control System (FCS) 
            g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
          """
Role = f"""

        You are an experienced Aerospace Engineer with 15+ years in aircraft and spacecraft design.
        Talk in clear, direct, practical language — no fluff.
        Use simple words when possible. Avoid heavy jargon unless necessary (and then explain it briefly).
        
            Main characteristics:

                Answer technically but keep it understandable
                    -Prefer bullet points • numbered lists when explaining
                    -Give realistic numbers / rules of thumb when asked
                    -Say “I don’t know” or “need more data” when something is unclear
                    -Focus on physics, aerodynamics, structures, propulsion, flight mechanics, materials, testing
                    -Always think about weight • cost • safety • performance trade-offs
                    -Prefer metric units (kg, m, kN, Pa, °C), add imperial in brackets if useful

        """


response = ollama.chat(
    model='gemma3:4b', # ← Tells Ollama: "use this specific model"
    options={
        'temperature': 0.99,
    },
                                  
    messages=[
        {
            'role': 'system',
            'content': Role 
        },
        {
            'role': 'user',       # ← Pretends this message comes from the "human/user"
            'content': question   # ← The actual text we send = just the question
        }
        
    ]
)


print(response['message']['content'])
del response

Okay, let’s break down the F/A-18 Hornet’s FCS g-limiter and the 7.5g limit.

Here's the key information:

*   **The Limit:** The FCS g-limiter is designed to prevent exceeding certain g-forces during maneuvers. A commanded symmetric positive load factor (NzREF – that's the force acting on the aircraft’s center of gravity, perpendicular to the roll axis, due to the control system’s commands) of 7.5g is the maximum it will allow in a stable, controlled situation.

*   **Weight Dependence:** The 7.5g limit isn’t a fixed number. It’s directly tied to the aircraft’s gross weight. The heavier the aircraft, the lower the g-force it can withstand without the FCS needing to actively intervene. 

*   **Specific Weight Range:** For the F/A-18 Hornet, the FCS g-limiter permits a maximum commanded symmetric positive load factor (NzREF) of 7.5g between approximately **18,800 kg (41,650 lbs)** and **21,400 kg (47,200 lbs)** gross weight. 

    *   **Why the Range?** The FCS uses sensors and algorith

### For comparison !

#### From NATOPS FLIGHT MANUAL NAVY MODEL F/A-18A/B/C/D 161353 AND UP AIRCRAFT
    
    11.1.7 G−Limiter. The g−limiter function in the FCS limits commanded load factor under most
           flight conditions to the symmetric load limit (NzREF) based on gross weight below 
           44,000 pounds gross weight (maximum NzREF limit of 7.5 g). Above 44,000 pounds, 
           NzREF is held constant at 5.5 g. The negative load factor limit command is fixed at 
           negative 3 g’s for all gross weights. At weights greater than 32,357 pounds the 
           g−limiter does not provide adequate negative g protection and aircraft overstress 
           is possible.

    Correct ? Yes/no ?

## 🧪 Experiment 2: "Prompt Stuffing" (The Messy Way)

    Now, let's try to give the AI the information by "stuffing" it all into the prompt.
    
        The Problem: If your document is 1,000 pages long, you can't fit it in the 
        chat box. It's too expensive and the AI gets "lost in the middle."

### Read a PDF

In [3]:
import fitz

pdf_path = "F18-ABCD-000.pdf"


doc = fitz.open(pdf_path)
print(f"Opened PDF with {len(doc)} pages\n")

# We'll store everything here
PDF = ""

for i in range(len(doc)):
    page = doc[i]
    text = page.get_text("text")  # "text" mode gives "natural" reading order

    # Optional: add clear page separators
    PDF += f"\n{'═' * 80}\n"
    PDF += f"Page {i+1:3d} of {len(doc)}\n"
    PDF += f"{'═' * 80}\n\n"

    if text.strip():
        PDF += text + "\n\n"
    else:
        PDF += "[No extractable text on this page]\n\n"

doc.close()


Opened PDF with 902 pages



In [4]:
print(PDF)


════════════════════════════════════════════════════════════════════════════════
Page   1 of 902
════════════════════════════════════════════════════════════════════════════════

NATOPS FLIGHT MANUAL
NAVY MODEL
F/A-18A/B/C/D
161353 AND UP
AIRCRAFT
THIS PUBLICATION SUPERSEDES A1-F18AC-NFM-000 DATED
1 AUGUST 2006 CHANGED 15 JUNE 2007
THIS PUBLICATION IS INCOMPLETE WITHOUT
A1-F18AC-NFM-200 and A1-F18AC-NFM-210
DISTRIBUTION STATEMENT C. Distribution authorized to U.S. Government
agencies only and their contractors to protect publications required for official
use or for administrative or operational purposes only, determined on
1 August 2006. Other requests for this document shall be referred to
Commander, Naval Air Systems Command (PMA-265), RADM William A.
Moffett Bldg, 47123 Buse Rd, Bldg 2272, Patuxent River, MD 20670-1547.
DESTRUCTION NOTICE - For unclassified, limited documents, destroy by any
method that will prevent disclosure of contents or reconstruction of the
document.
ISSUED 

In [5]:
import ollama

question = f"""
            For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
            g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
            """
prompt = f"""
            You are a precise technical assistant. Answer ONLY using the provided CONTEXT.

            QUESTION:
            {question}

            CONTEXT:
            {PDF}

            Rules — you MUST follow all of them:
                1. Use ONLY facts explicitly stated in the CONTEXT.
                2. Do NOT use any outside knowledge, even if it is common aerospace knowledge.
                3. Do NOT infer, extrapolate, assume or "fill in gaps".
                4. If the CONTEXT lacks the necessary information → reply only with:  
                       "Not enough information in the provided context."
                5. Keep answers short and factual.
                6. Use bullet points for lists, tables or comparisons.
                7. Preserve original units and terminology from the context.

            Answer:
        
          """

Role = f"""

        You are an experienced Aerospace Engineer with 15+ years in aircraft and spacecraft design.
        Talk in clear, direct, practical language — no fluff.
        Use simple words when possible. Avoid heavy jargon unless necessary (and then explain it briefly).
        
            Main characteristics:

                Answer technically but keep it understandable
                    -Prefer bullet points • numbered lists when explaining
                    -Give realistic numbers / rules of thumb when asked
                    -Say “I don’t know” or “need more data” when something is unclear
                    -Focus on physics, aerodynamics, structures, propulsion, flight mechanics, materials, testing
                    -Always think about weight • cost • safety • performance trade-offs
                    -Prefer metric units (kg, m, kN, Pa, °C), add imperial in brackets if useful

        """


response = ollama.chat(
    model='gemma3:4b',
    options={
        'temperature': 0.99,
    },
    messages=[
        {
            'role': 'system',
            'content': Role 
        },
        {
            'role': 'user',
            'content': prompt
        }
    ]
)

print(response['message']['content'])

# Optional: clean up
del response

Not enough information in the provided context.


## 🏗️ The 6-Stage RAG Pipeline

    RAG is the "Smart Way." Instead of sending the whole book, we only send the relevant paragraph.

    Step	
        1. Load	Read your PDF/Text	
        2. Split	Cut text into small pieces	
        3. Embed	Turn text into "Meaning Numbers"	
        4. Store	Put numbers in a Vector DB	
        5. Retrieve	Find chunks that match the question	
        6. Generate	AI writes the final answer	

### Load,Split,Embed and Store

In [6]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 1. Setup paths and models
pdf_path = "F18-ABCD-000.pdf"
persist_directory = "./chroma_db"

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize the splitter (prevents pages from being too long for the embedding model)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 2. Extract and Process Page by Page
doc = fitz.open(pdf_path)
print(f"Opened PDF with {len(doc)} pages\n")

all_page_documents = []

for i in range(len(doc)):
    page = doc[i]
    text = page.get_text("text")

    if text.strip():
        # Split text of the current page into manageable chunks
        page_chunks = text_splitter.split_text(text)
        
        # Convert each chunk into a LangChain Document object with metadata
        for chunk in page_chunks:
            new_doc = Document(
                page_content=chunk,
                metadata={"page": i + 1, "source": pdf_path}
            )
            all_page_documents.append(new_doc)
    else:
        print(f"Skipping page {i+1}: No extractable text.")

doc.close()

# 3. Create and Persist the Vector Database
print(f"Encoding {len(all_page_documents)} chunks into the vector database...")

vector_db = Chroma.from_documents(
    documents=all_page_documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("Vector database created and saved locally.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Opened PDF with 902 pages

Skipping page 4: No extractable text.
Skipping page 6: No extractable text.
Skipping page 34: No extractable text.
Skipping page 36: No extractable text.
Skipping page 63: No extractable text.
Skipping page 66: No extractable text.
Skipping page 72: No extractable text.
Skipping page 284: No extractable text.
Skipping page 298: No extractable text.
Skipping page 302: No extractable text.
Skipping page 306: No extractable text.
Skipping page 346: No extractable text.
Skipping page 382: No extractable text.
Skipping page 432: No extractable text.
Skipping page 466: No extractable text.
Skipping page 468: No extractable text.
Skipping page 538: No extractable text.
Skipping page 604: No extractable text.
Skipping page 626: No extractable text.
Skipping page 666: No extractable text.
Skipping page 672: No extractable text.
Skipping page 674: No extractable text.
Skipping page 682: No extractable text.
Skipping page 686: No extractable text.
Skipping page 788: No 

### Retrieve

In [11]:
query = """
        For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
        g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
        """
docs = vector_db.similarity_search(query, k=3)

for d in docs:
    print(f"Content: {d.page_content}...")
    print(f"Source: Page {d.metadata['page']}\n")

Content: 11.1.7
G−Limiter.
The g−limiter function in the FCS limits commanded load factor under most
flight conditions to the symmetric load limit (NzREF) based on gross weight below 44,000 pounds gross
weight (maximum NzREF limit of 7.5 g). Above 44,000 pounds, NzREF is held constant at 5.5 g. The
negative load factor limit command is fixed at negative 3 g’s for all gross weights. At weights greater
than 32,357 pounds the g−limiter does not provide adequate negative g protection and aircraft
overstress is possible.
During rolling maneuvers, the g−limiter reduces commanded load factor up to 80% NzREF. The
additional reduction begins with 0.75 inch lateral stick up to 80% at full lateral stick input of 3 inches.
Very abrupt stick commands can exceed the capabilities of the system and result in an overstress. The
g−limiter can be disengaged during emergency situations by pressing the paddle switch to allow 33%...
Source: Page 436

Content: 2.8.2.3 G Limiter.
The g limiter prevents exceed

### Generate

In [12]:
import ollama

question = f"""
            For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
            g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
            """
prompt = f"""
            You are a precise technical assistant. Answer ONLY using the provided CONTEXT.

            QUESTION:
            {question}

            CONTEXT:
            {docs}

            Rules — you MUST follow all of them:
                1. Use ONLY facts explicitly stated in the CONTEXT.
                2. Do NOT use any outside knowledge, even if it is common aerospace knowledge.
                3. Do NOT infer, extrapolate, assume or "fill in gaps".
                4. If the CONTEXT lacks the necessary information → reply only with:  
                       "Not enough information in the provided context."
                5. Keep answers short and factual.
                6. Use bullet points for lists, tables or comparisons.
                7. Preserve original units and terminology from the context.

            Answer:
        
          """

Role = f"""

        You are an experienced Aerospace Engineer with 15+ years in aircraft and spacecraft design.
        Talk in clear, direct, practical language — no fluff.
        Use simple words when possible. Avoid heavy jargon unless necessary (and then explain it briefly).
        
            Main characteristics:

                Answer technically but keep it understandable
                    -Prefer bullet points • numbered lists when explaining
                    -Give realistic numbers / rules of thumb when asked
                    -Say “I don’t know” or “need more data” when something is unclear
                    -Focus on physics, aerodynamics, structures, propulsion, flight mechanics, materials, testing
                    -Always think about weight • cost • safety • performance trade-offs
                    -Prefer metric units (kg, m, kN, Pa, °C), add imperial in brackets if useful

        """


response = ollama.chat(
    model='gemma3:4b',
    options={
        'temperature': 0.99,
    },
    messages=[
        {
            'role': 'system',
            'content': Role 
        },
        {
            'role': 'user',
            'content': prompt
        }
    ]
)

print(response['message']['content'])

# Optional: clean up
del response

Below 44,000 pounds gross weight, the maximum commanded symmetric positive load factor (NzREF) is 7.5 g.
